# A9 Batch Inference & Aggregation (CPU)

Menjalankan full-corpus inference dan aggregation pada canonical reviews
menggunakan model TF-IDF yang terpilih setelah A8, melalui
`sipature_ml.a9.run_inference` dan `run_aggregation`.
Ikuti `docs/a9-inference-priority-report.md`.

Input: `data/processed/canonical_reviews.parquet` (notebook `02`) dan
`models/tfidf-aspect-silver-v1/` (notebook `05`).
Output: `a9/<run>-infer/` dan `a9/<run>-aggregate/` di Drive.

Output level-review bersifat **restricted** (tidak dicopy ke repo).


## Step 1 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Konfigurasi path & parameter


In [ ]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from datetime import datetime, timezone
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")

# Input (hasil notebook 02 dan notebook 05).
REVIEWS_PATH = DRIVE_ROOT / "data" / "processed" / "canonical_reviews.parquet"
MODEL_DIR = DRIVE_ROOT / "models" / "tfidf-aspect-silver-v1"

# Output immutable (timestamp agar tidak menimpa run lama).
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M_a9-tfidf-lexical-v1")
INFERENCE_DIR = DRIVE_ROOT / "a9" / f"{RUN_ID}-infer"
AGGREGATION_DIR = DRIVE_ROOT / "a9" / f"{RUN_ID}-aggregate"

PROJECT_DIR = Path("/content/hackathon/ml")

print("Reviews:", REVIEWS_PATH)
print("Model dir:", MODEL_DIR)
print("Run ID:", RUN_ID)
print("Inference dir:", INFERENCE_DIR)
print("Aggregation dir:", AGGREGATION_DIR)


## Step 3 — Clone repository dari GitHub


In [ ]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)
assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


## Step 4 — Verifikasi commit terbaru (git log)


In [ ]:
%cd /content/hackathon/ml
!git log --oneline -3


## Step 5 — Install dependencies (CPU profile)


In [ ]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-dev.lock.txt
!python -m pip install --no-deps -e .


**RESTART WAJIB.** Setelah install, restart runtime agar numpy/sklearn lama
tidak ter-cache di memori.

1. **Runtime > Restart session**
2. Jalankan ulang **Step 1** (mount) dan **Step 2** (config)
3. Step 3–5 **tidak perlu diulang**

Lalu lanjut ke **Step 6**.


## Step 6 — Verifikasi versi package (setelah restart)


In [ ]:
import joblib
import numpy
import pandas
import pyarrow
import sklearn

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

assert sklearn.__version__ == "1.7.2", (
    f"scikit-learn 1.7.2 diperlukan untuk memuat model TF-IDF, "
    f"ditemukan {sklearn.__version__}"
)
print("\nEnvironment A9 siap.")


## Step 7 — Import modul sipature_ml


In [ ]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"
assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml
print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


## Step 8 — Load config A9 & verifikasi kontrak model TF-IDF


In [ ]:
from sipature_ml.a9 import load_tfidf_contract
from sipature_ml.config import load_config

config = load_config("a9")

print("A9 version:", config["a9_version"])
print("Aspect model:", config["models"]["aspect"]["version"])
print("Polarity model:", config["models"]["polarity"]["version"])
print("Severity status:", config["models"]["severity"]["status"])

artifact = load_tfidf_contract(MODEL_DIR, config)
print("\nModel TF-IDF terverifikasi.")
print("Jumlah aspek:", len(artifact["aspects"]))
print("Aspek:", artifact["aspects"])
print("Thresholds:", [round(t, 3) for t in artifact["thresholds"]])


## Step 9 — Jalankan full-corpus inference


In [ ]:
from sipature_ml.a9 import run_inference

assert not INFERENCE_DIR.exists(), f"Sudah ada: {INFERENCE_DIR}"

summary = run_inference(
    reviews_path=REVIEWS_PATH,
    model_dir=MODEL_DIR,
    output_dir=INFERENCE_DIR,
)

print("A9 version:", summary["a9_version"])
print("Text reviews:", summary["text_reviews"])
print("Reviews with predictions:", summary["reviews_with_predictions"])
print("Aspect predictions:", summary["aspect_predictions"])
print("Aspect model:", summary["aspect_model"])
print("Polarity model:", summary["polarity_model"])
print("Restricted:", summary["restricted"])


## Step 10 — Jalankan aggregation (destination-aspect signals + evidence)


In [ ]:
from sipature_ml.a9 import run_aggregation

assert not AGGREGATION_DIR.exists(), f"Sudah ada: {AGGREGATION_DIR}"

summary = run_aggregation(
    predictions_dir=INFERENCE_DIR,
    reviews_path=REVIEWS_PATH,
    output_dir=AGGREGATION_DIR,
)

print("A9 version:", summary["a9_version"])
print("Signals:", summary["signals"])
print("Evidence items:", summary["evidence_items"])
print("Destinations:", summary["destinations"])
print("Severity status:", summary["severity_status"])
print("Restricted:", summary["restricted"])


## Step 11 — Verifikasi output & hash artifact


In [ ]:
import json
from pathlib import Path

from sipature_ml.manifest import sha256_file

for name, directory in (("infer", INFERENCE_DIR), ("aggregate", AGGREGATION_DIR)):
    manifest = json.loads(
        (directory / "manifest.json").read_text(encoding="utf-8")
    )
    print(f"=== {name} ===")
    print("  Stage:", manifest["stage"])
    print("  A9 version:", manifest["a9_version"])
    errors = []
    for relative, expected in manifest["artifact_hashes"].items():
        path = directory / relative
        if not path.is_file():
            errors.append(f"missing: {relative}")
        elif sha256_file(path) != expected:
            errors.append(f"hash mismatch: {relative}")
    print(f"  Artifact check: {len(manifest['artifact_hashes'])} file, {len(errors)} masalah")
    assert not errors, errors

print("\nSeluruh output A9 valid terhadap manifest.")


## Step 12 — Run summary


In [ ]:
print("RUN ID:", RUN_ID)
print("INFERENCE DIR :", INFERENCE_DIR)
print("AGGREGATION DIR:", AGGREGATION_DIR)
print("REVIEWS       :", REVIEWS_PATH)
print("MODEL DIR     :", MODEL_DIR)

print("\nREMINDER: output review-level (predictions/evidence) bersifat restricted.")
print("Lanjut ke notebook 09 untuk prioritization + export (privacy-safe).")
